In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os

# 1. 나눔 폰트 파일 직접 찾기
font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

# 2. 파일이 존재하는지 확인
if os.path.exists(font_path):
    # 3. 폰트 등록 및 사용
    fm.fontManager.addfont(font_path)
    font_prop = fm.FontProperties(fname=font_path)
    
    # 4. 전역 설정
    plt.rc('font', family=font_prop.get_name())
    plt.rcParams['axes.unicode_minus'] = False
    
    print(f"✅ Font set to: {font_prop.get_name()}")
else:
    print(f"❌ Font file not found at: {font_path}")
    print("\nLet's find where the fonts are...")
    
    # 폰트 찾기
    import subprocess
    result = subprocess.run(['find', '/usr', '-name', '*NanumGothic*.ttf'], 
                          capture_output=True, text=True)
    if result.stdout:
        print("Found fonts at:")
        print(result.stdout)

# 시스템에 설치된 모든 폰트 확인
fonts = [f.name for f in fm.fontManager.ttflist]

# 한글 폰트 찾기
korean_fonts = [font for font in fonts if 'Nanum' in font or 'Gothic' in font or 'Gulim' in font or 'Batang' in font]
print("Available Korean fonts:", korean_fonts)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path
import warnings
import os
import matplotlib.font_manager as fm
warnings.filterwarnings('ignore')

# 한글 폰트 설정 (간단 버전)
def setup_korean_font():
    """Simple Korean font setup"""
    font_paths = [
        '/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
        '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf',
        '/usr/share/fonts/truetype/nanum/NanumMyeongjo.ttf',
        '/usr/local/share/fonts/NanumGothic.ttf',
    ]
    
    for font_path in font_paths:
        if os.path.exists(font_path):
            try:
                fm.fontManager.addfont(font_path)
                font_prop = fm.FontProperties(fname=font_path)
                plt.rc('font', family=font_prop.get_name())
                plt.rcParams['axes.unicode_minus'] = False
                print(f"✓ Korean font set: {font_prop.get_name()}")
                return True
            except Exception as e:
                continue
    
    print("⚠ Korean font not found. Using default font.")
    plt.rcParams['axes.unicode_minus'] = False
    return False

# 한글 폰트 설정 실행
korean_font_available = setup_korean_font()

# 파스텔톤 색상 정의
pastel_colors = ['#FFB3BA', '#BAFFC9', '#BAE1FF', '#FFFFBA', '#FFDFBA', '#E0BBE4']

def find_time_column(df):
    """Find time/datetime column in dataframe"""
    time_column_candidates = ['timestamp', 'datetime', 'time', 'date', '시간', '날짜', 
                              'Timestamp', 'DateTime', 'Time', 'Date']
    
    for col in df.columns:
        if col in time_column_candidates or 'time' in col.lower() or '시간' in col:
            return col
    return None

def plot_column_distributions_separate(folder_path, column_name, time_column=None, figsize=(12, 12)):
    """
    Plot distributions of a specific column from multiple CSV files in separate subplots
    
    Parameters:
    -----------
    folder_path : str
        Folder path containing CSV files
    column_name : str
        Column name to analyze
    time_column : str, optional
        Name of time column. If None, will auto-detect
    figsize : tuple
        Figure size
    """
    
    # CSV files to find
    csv_files = ['livingroom.csv', 'room1.csv', 'room2.csv']
    
    # Dictionary for data storage
    data_dict = {}
    time_info_dict = {}
    
    # Read each CSV file
    for csv_file in csv_files:
        file_path = Path(folder_path) / csv_file
        if file_path.exists():
            try:
                df = pd.read_csv(file_path)
                
                # Auto-detect time column if not specified
                detected_time_col = time_column if time_column else find_time_column(df)
                
                if column_name in df.columns:
                    # Create a working copy with reset index
                    df_work = df[[column_name]].copy()
                    
                    # Parse time column if available
                    if detected_time_col and detected_time_col in df.columns:
                        try:
                            df_work['_time'] = pd.to_datetime(df[detected_time_col], errors='coerce')
                            print(f"  ⏰ Time column detected: '{detected_time_col}'")
                        except:
                            print(f"  ⚠ Could not parse time column: '{detected_time_col}'")
                            df_work['_time'] = None
                    else:
                        df_work['_time'] = None
                    
                    # Convert to numeric
                    df_work[column_name] = pd.to_numeric(df_work[column_name], errors='coerce')
                    
                    # Remove rows with NaN values in the target column
                    df_work = df_work.dropna(subset=[column_name])
                    
                    # Remove extreme outliers
                    if len(df_work) > 0:
                        data = df_work[column_name]
                        q1 = data.quantile(0.0)
                        q3 = data.quantile(0.99)
                        iqr = q3 - q1
                        lower_bound = q1 - 3 * iqr
                        upper_bound = q3 + 3 * iqr
                        
                        # Filter outliers
                        df_work = df_work[(data >= lower_bound) & (data <= upper_bound)]
                        
                        # Check if too many values were removed
                        removed_count = len(df) - len(df_work)
                        if removed_count > 0:
                            print(f"  ⚠ Removed {removed_count} outliers/invalid from {csv_file}")
                    
                    if len(df_work) > 0:
                        data = df_work[column_name]
                        time_data = df_work['_time'] if '_time' in df_work.columns else None
                        
                        data_dict[csv_file.replace('.csv', '')] = data
                        
                        # Store time information for each unique value
                        if time_data is not None and not time_data.isna().all():
                            value_time_ranges = {}
                            for value in data.unique():
                                value_indices = (data == value)
                                value_times = time_data[value_indices].dropna()
                                if len(value_times) > 0:
                                    value_time_ranges[value] = {
                                        'first': value_times.min(),
                                        'last': value_times.max(),
                                        'count': len(value_times)
                                    }
                            time_info_dict[csv_file.replace('.csv', '')] = value_time_ranges
                        
                        print(f"✓ {csv_file} loaded successfully")
                        print(f"  - Total rows: {len(df):,}")
                        print(f"  - Valid numeric values: {len(data):,}")
                        print(f"  - Invalid/removed: {len(df) - len(data):,} ({(len(df) - len(data))/len(df)*100:.1f}%)")
                    else:
                        print(f"⚠ No valid numeric data in column '{column_name}' in {csv_file}")
                else:
                    print(f"⚠ Column '{column_name}' not found in {csv_file}")
            except Exception as e:
                print(f"⚠ Error reading {csv_file}: {e}")
                import traceback
                traceback.print_exc()
        else:
            print(f"⚠ File {csv_file} not found")
    
    if not data_dict:
        print("No data available for analysis")
        return
    
    # Create subplots: 3 rows (one for each room), 2 columns (histogram, boxplot)
    n_rooms = len(data_dict)
    fig, axes = plt.subplots(n_rooms, 2, figsize=figsize)
    
    # If only one room, make it 2D array
    if n_rooms == 1:
        axes = axes.reshape(1, -1)
    
    # Plot for each room
    for idx, (room, data) in enumerate(data_dict.items()):
        color = pastel_colors[idx % len(pastel_colors)]
        
        # 1. Histogram with count information
        ax1 = axes[idx, 0]
        ax1.hist(data, alpha=0.7, color=color, bins=30, edgecolor='white', linewidth=0.5)
        ax1.axvline(data.mean(), color='red', linestyle='--', linewidth=2, alpha=0.7, label=f'Mean: {data.mean():.2f}')
        ax1.axvline(data.median(), color='blue', linestyle='--', linewidth=2, alpha=0.7, label=f'Median: {data.median():.2f}')
        
        # Add total count and unique count to the plot title
        unique_count = data.nunique()
        ax1.set_title(f'Histogram (N={len(data):,}, Unique={unique_count:,})' if idx == 0 else f'(N={len(data):,}, Unique={unique_count:,})', 
                     fontsize=11, fontweight='bold')
        ax1.set_xlabel(column_name if idx == n_rooms-1 else '', fontsize=10)
        ax1.set_ylabel(f'{room}\nFrequency', fontsize=10)
        ax1.legend(fontsize=8, loc='upper right')
        ax1.grid(True, alpha=0.3, linestyle='--')
        
        # 2. Boxplot with enhanced statistics
        ax2 = axes[idx, 1]
        box = ax2.boxplot([data.values], widths=0.7, patch_artist=True,
                          boxprops=dict(linewidth=1.5, facecolor=color, alpha=0.8),
                          medianprops=dict(color='#666666', linewidth=2),
                          whiskerprops=dict(linewidth=1.5),
                          capprops=dict(linewidth=1.5))
        
        # Add comprehensive stats text with value frequency info
        mode_value = data.mode()[0] if len(data.mode()) > 0 else data.mean()
        mode_count = (data == mode_value).sum()
        
        stats_text = (f'Total Count: {len(data):,}\n'
                     f'Unique Values: {unique_count:,}\n'
                     f'Unique Ratio: {unique_count/len(data)*100:.1f}%\n'
                     f'Most Frequent: {mode_value:.1f} ({mode_count:,}x)\n'
                     f'Mean: {data.mean():.2f}\n'
                     f'Std: {data.std():.2f}\n'
                     f'Q1: {data.quantile(0.25):.2f}\n'
                     f'Q3: {data.quantile(0.75):.2f}\n'
                     f'IQR: {data.quantile(0.75) - data.quantile(0.25):.2f}')
        ax2.text(1.3, data.median(), stats_text, fontsize=8, verticalalignment='center',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        
        ax2.set_ylabel(column_name if idx == n_rooms-1 else '', fontsize=10)
        ax2.set_xticklabels([f'{room}\n(N={len(data):,})'])
        ax2.set_title('Boxplot' if idx == 0 else '', fontsize=11, fontweight='bold')
        ax2.grid(True, alpha=0.3, axis='y', linestyle='--')
    
    plt.suptitle(f'Distribution Analysis: "{column_name}" by Room', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    # Statistical summary with enhanced count information
    print("\n📊 Statistical Summary:")
    print("=" * 110)
    print(f"{'Room':<15} {'Total':<10} {'Unique':<10} {'Unique %':<10} {'Mean':<10} {'Median':<10} {'Std':<10} {'Min':<10} {'Max':<10}")
    print("=" * 110)
    
    total_all_rooms = sum(len(data) for data in data_dict.values())
    total_unique_all = sum(data.nunique() for data in data_dict.values())
    
    for room, data in data_dict.items():
        unique_count = data.nunique()
        unique_pct = (unique_count / len(data)) * 100
        print(f"{room:<15} {len(data):<10,} {unique_count:<10,} {unique_pct:<10.1f} {data.mean():<10.2f} {data.median():<10.2f} "
              f"{data.std():<10.2f} {data.min():<10.2f} {data.max():<10.2f}")
    
    print("=" * 110)
    print(f"{'TOTAL':<15} {total_all_rooms:<10,} {total_unique_all:<10,}")
    print("=" * 110)
    
    # Print unique value details for each room
    print("\n🔢 Unique Value Details:")
    print("=" * 60)
    for room, data in data_dict.items():
        unique_count = data.nunique()
        unique_ratio = (unique_count / len(data)) * 100
        print(f"{room}:")
        print(f"  - Total data points: {len(data):,}")
        print(f"  - Unique values: {unique_count:,}")
        print(f"  - Unique ratio: {unique_ratio:.2f}%")
        print(f"  - Average repetition: {len(data)/unique_count:.2f}x per value")
        
        # Show top 5 most frequent values with time ranges
        value_counts = data.value_counts().head(5)
        print(f"  - Top 5 most frequent values:")
        
        has_time_info = room in time_info_dict
        
        for val, count in value_counts.items():
            freq_pct = count/len(data)*100
            base_info = f"      {val:.2f}: {count:,} times ({freq_pct:.2f}%)"
            
            if has_time_info and val in time_info_dict[room]:
                time_range = time_info_dict[room][val]
                first_time = time_range['first'].strftime('%Y-%m-%d %H:%M:%S')
                last_time = time_range['last'].strftime('%Y-%m-%d %H:%M:%S')
                duration = time_range['last'] - time_range['first']
                print(f"{base_info}")
                print(f"          ⏰ Time range: {first_time} ~ {last_time}")
                print(f"          ⏱️  Duration: {duration}")
            else:
                print(base_info)
        print()
    print("=" * 60)
    
    # Additional detailed time range report if time info available
    if time_info_dict:
        print("\n⏰ Detailed Time Range Analysis:")
        print("=" * 100)
        for room, value_time_ranges in time_info_dict.items():
            print(f"\n{room.upper()}:")
            print("-" * 100)
            print(f"{'Value':<12} {'Count':<10} {'First Appearance':<25} {'Last Appearance':<25} {'Duration':<15}")
            print("-" * 100)
            
            # Sort by count (descending) and show top 10
            sorted_values = sorted(value_time_ranges.items(), 
                                 key=lambda x: x[1]['count'], 
                                 reverse=True)[:10]
            
            for value, time_info in sorted_values:
                first_str = time_info['first'].strftime('%Y-%m-%d %H:%M:%S')
                last_str = time_info['last'].strftime('%Y-%m-%d %H:%M:%S')
                duration = time_info['last'] - time_info['first']
                print(f"{value:<12.2f} {time_info['count']:<10} {first_str:<25} {last_str:<25} {str(duration):<15}")
            print("-" * 100)
        print("=" * 100)
    
    plt.show()
    
    return fig

def plot_column_comparison(folder_path, column_name, time_column=None, figsize=(15, 5)):
    """
    Compare distributions across rooms in a single row
    
    Parameters:
    -----------
    folder_path : str
        Folder path containing CSV files
    column_name : str
        Column name to analyze
    time_column : str, optional
        Name of time column. If None, will auto-detect
    figsize : tuple
        Figure size
    """
    
    # CSV files to find
    csv_files = ['livingroom.csv', 'room1.csv', 'room2.csv']
    
    # Dictionary for data storage
    data_dict = {}
    time_info_dict = {}
    
    # Read each CSV file
    for csv_file in csv_files:
        file_path = Path(folder_path) / csv_file
        if file_path.exists():
            try:
                df = pd.read_csv(file_path)
                
                # Auto-detect time column if not specified
                detected_time_col = time_column if time_column else find_time_column(df)
                
                if column_name in df.columns:
                    # Create a working copy
                    df_work = df[[column_name]].copy()
                    
                    # Parse time column if available
                    if detected_time_col and detected_time_col in df.columns:
                        try:
                            df_work['_time'] = pd.to_datetime(df[detected_time_col], errors='coerce')
                        except:
                            df_work['_time'] = None
                    else:
                        df_work['_time'] = None
                    
                    # Convert to numeric
                    df_work[column_name] = pd.to_numeric(df_work[column_name], errors='coerce')
                    
                    # Remove rows with NaN values in the target column
                    df_work = df_work.dropna(subset=[column_name])
                    
                    # Remove extreme outliers
                    if len(df_work) > 0:
                        data = df_work[column_name]
                        q1 = data.quantile(0.0)
                        q3 = data.quantile(0.99)
                        iqr = q3 - q1
                        lower_bound = q1 - 3 * iqr
                        upper_bound = q3 + 3 * iqr
                        
                        # Filter outliers
                        df_work = df_work[(data >= lower_bound) & (data <= upper_bound)]
                        
                        # Check if too many values were removed
                        removed_count = len(df) - len(df_work)
                        if removed_count > 0:
                            print(f"  ⚠ Removed {removed_count} outliers/invalid from {csv_file}")
                    
                    if len(df_work) > 0:
                        data = df_work[column_name]
                        time_data = df_work['_time'] if '_time' in df_work.columns else None
                        unique_count = data.nunique()
                        
                        data_dict[csv_file.replace('.csv', '')] = data
                        
                        # Store time information for each unique value
                        if time_data is not None and not time_data.isna().all():
                            value_time_ranges = {}
                            for value in data.unique():
                                value_indices = (data == value)
                                value_times = time_data[value_indices].dropna()
                                if len(value_times) > 0:
                                    value_time_ranges[value] = {
                                        'first': value_times.min(),
                                        'last': value_times.max(),
                                        'count': len(value_times)
                                    }
                            time_info_dict[csv_file.replace('.csv', '')] = value_time_ranges
                        
                        print(f"✓ {csv_file} loaded successfully ({len(df)} rows, {len(data)} valid values, {unique_count} unique)")
                    else:
                        print(f"⚠ No valid numeric data in column '{column_name}' in {csv_file}")
                else:
                    print(f"⚠ Column '{column_name}' not found in {csv_file}")
            except Exception as e:
                print(f"⚠ Error reading {csv_file}: {e}")
                import traceback
                traceback.print_exc()
        else:
            print(f"⚠ File {csv_file} not found")
    
    if not data_dict:
        print("No data available for analysis")
        return
    
    # Create subplots: 1 row, 3 columns (one for each room)
    n_rooms = len(data_dict)
    fig, axes = plt.subplots(1, n_rooms, figsize=figsize)
    
    # If only one room, make it array
    if n_rooms == 1:
        axes = [axes]
    
    # Plot for each room
    for idx, (room, data) in enumerate(data_dict.items()):
        ax = axes[idx]
        color = pastel_colors[idx % len(pastel_colors)]
        
        # Histogram
        n, bins, patches = ax.hist(data, bins=30, alpha=0.7, color=color, 
                                   edgecolor='white', linewidth=0.5)
        
        # Add statistics lines
        ax.axvline(data.mean(), color='red', linestyle='--', linewidth=2, alpha=0.7, label=f'Mean: {data.mean():.1f}')
        ax.axvline(data.median(), color='blue', linestyle=':', linewidth=2, alpha=0.7, label=f'Median: {data.median():.1f}')
        
        # Labels and formatting with total count and unique count
        unique_count = data.nunique()
        ax.set_title(f'{room.upper()}\nN: {len(data):,} | Unique: {unique_count:,}', fontsize=12, fontweight='bold')
        ax.set_xlabel(column_name, fontsize=10)
        ax.set_ylabel('Frequency' if idx == 0 else '', fontsize=10)
        ax.legend(fontsize=8, loc='upper right')
        ax.grid(True, alpha=0.3, linestyle='--')
        
        # Add comprehensive stats box with unique value count
        mode_value = data.mode()[0] if len(data.mode()) > 0 else data.mean()
        mode_count = (data == mode_value).sum()
        
        stats_text = (f'Total: {len(data):,}\n'
                     f'Unique: {unique_count:,}\n'
                     f'Unique %: {unique_count/len(data)*100:.1f}%\n'
                     f'Mode: {mode_value:.1f} ({mode_count:,}x)\n'
                     f'σ: {data.std():.2f}\n'
                     f'Min: {data.min():.1f}\n'
                     f'Max: {data.max():.1f}')
        ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, fontsize=8,
                verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
    
    plt.suptitle(f'Room Comparison: "{column_name}"', fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    # Print unique value comparison
    print("\n🔢 Unique Value Comparison:")
    print("=" * 80)
    print(f"{'Room':<15} {'Total':<12} {'Unique':<12} {'Unique %':<12} {'Avg Repeat':<12}")
    print("=" * 80)
    for room, data in data_dict.items():
        unique_count = data.nunique()
        unique_pct = (unique_count / len(data)) * 100
        avg_repeat = len(data) / unique_count
        print(f"{room:<15} {len(data):<12,} {unique_count:<12,} {unique_pct:<12.2f} {avg_repeat:<12.2f}x")
    print("=" * 80)
    
    # Time range information if available
    if time_info_dict:
        print("\n⏰ Time Range Summary (Top 5 per room):")
        print("=" * 100)
        for room, value_time_ranges in time_info_dict.items():
            print(f"\n{room.upper()}:")
            print("-" * 100)
            
            # Sort by count and show top 5
            sorted_values = sorted(value_time_ranges.items(), 
                                 key=lambda x: x[1]['count'], 
                                 reverse=True)[:5]
            
            for value, time_info in sorted_values:
                first_str = time_info['first'].strftime('%Y-%m-%d %H:%M:%S')
                last_str = time_info['last'].strftime('%Y-%m-%d %H:%M:%S')
                duration = time_info['last'] - time_info['first']
                print(f"  Value {value:.2f}: {time_info['count']:,} occurrences")
                print(f"    ⏰ {first_str} ~ {last_str} (Duration: {duration})")
        print("=" * 100)
    
    plt.show()
    
    return fig

def plot_multiple_columns_by_room(folder_path, column_names, time_column=None, figsize=(15, 12)):
    """
    Visualize multiple columns organized by room
    
    Parameters:
    -----------
    folder_path : str
        Folder path containing CSV files
    column_names : list
        List of column names to analyze
    time_column : str, optional
        Name of time column. If None, will auto-detect
    """
    
    csv_files = ['livingroom.csv', 'room1.csv', 'room2.csv']
    
    # Load data
    dataframes = {}
    time_columns = {}
    
    for csv_file in csv_files:
        file_path = Path(folder_path) / csv_file
        if file_path.exists():
            try:
                df = pd.read_csv(file_path)
                dataframes[csv_file.replace('.csv', '')] = df
                
                # Detect time column
                detected_time_col = time_column if time_column else find_time_column(df)
                if detected_time_col and detected_time_col in df.columns:
                    try:
                        time_columns[csv_file.replace('.csv', '')] = pd.to_datetime(df[detected_time_col], errors='coerce')
                    except:
                        pass
                
                print(f"✓ {csv_file} loaded successfully")
            except Exception as e:
                print(f"⚠ Error reading {csv_file}: {e}")
    
    if not dataframes:
        print("Unable to load data")
        return
    
    # Create subplots: rooms x columns
    n_rooms = len(dataframes)
    n_cols = len(column_names)
    
    fig, axes = plt.subplots(n_rooms, n_cols, figsize=figsize)
    
    # Handle single row or column cases
    if n_rooms == 1 and n_cols == 1:
        axes = np.array([[axes]])
    elif n_rooms == 1:
        axes = axes.reshape(1, -1)
    elif n_cols == 1:
        axes = axes.reshape(-1, 1)
    
    # Track unique values for reporting
    unique_value_report = {col: {} for col in column_names}
    time_range_report = {col: {} for col in column_names}
    
    for room_idx, (room, df) in enumerate(dataframes.items()):
        time_data = time_columns.get(room)
        
        for col_idx, column_name in enumerate(column_names):
            ax = axes[room_idx, col_idx]
            color = pastel_colors[(room_idx + col_idx) % len(pastel_colors)]
            
            if column_name in df.columns:
                # Create working dataframe
                df_work = df[[column_name]].copy()
                if time_data is not None:
                    df_work['_time'] = time_data.copy()
                
                # Convert to numeric and drop NaN
                df_work[column_name] = pd.to_numeric(df_work[column_name], errors='coerce')
                df_work = df_work.dropna(subset=[column_name])
                
                data = df_work[column_name]
                
                if len(data) > 0:
                    # Histogram
                    ax.hist(data, bins=30, alpha=0.7, color=color,
                           edgecolor='white', linewidth=0.5)
                    
                    # Mean line
                    mean_val = data.mean()
                    ax.axvline(mean_val, color='red', linestyle='--', linewidth=1.5, 
                              alpha=0.7, label=f'μ={mean_val:.1f}')
                    
                    # Track unique values
                    unique_count = data.nunique()
                    unique_value_report[column_name][room] = {
                        'total': len(data),
                        'unique': unique_count,
                        'unique_pct': (unique_count/len(data)*100)
                    }
                    
                    # Track time ranges if available
                    if '_time' in df_work.columns and not df_work['_time'].isna().all():
                        value_time_ranges = {}
                        for value in data.unique():
                            value_indices = (data == value)
                            value_times = df_work.loc[value_indices, '_time'].dropna()
                            if len(value_times) > 0:
                                value_time_ranges[value] = {
                                    'first': value_times.min(),
                                    'last': value_times.max(),
                                    'count': len(value_times)
                                }
                        if value_time_ranges:
                            if column_name not in time_range_report:
                                time_range_report[column_name] = {}
                            time_range_report[column_name][room] = value_time_ranges
                    
                    # Formatting with data count and unique count
                    if room_idx == 0:
                        ax.set_title(f'{column_name}\nN={len(data):,}, U={unique_count:,}', fontsize=10, fontweight='bold')
                    else:
                        ax.set_title(f'N={len(data):,}, U={unique_count:,}', fontsize=9)
                    if col_idx == 0:
                        ax.set_ylabel(f'{room}\nFrequency', fontsize=10)
                    ax.legend(fontsize=7, loc='upper right')
                    ax.grid(True, alpha=0.3, linestyle='--')
                else:
                    ax.text(0.5, 0.5, 'No Data', ha='center', va='center')
            else:
                ax.text(0.5, 0.5, 'Column Not Found', ha='center', va='center')
    
    plt.suptitle('Multi-Column Analysis by Room', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Print unique value report
    print("\n🔢 Unique Value Report by Column:")
    print("=" * 80)
    for column_name, room_data in unique_value_report.items():
        print(f"\n📊 {column_name}")
        print("-" * 80)
        print(f"{'Room':<15} {'Total':<12} {'Unique':<12} {'Unique %':<12}")
        print("-" * 80)
        for room, stats in room_data.items():
            print(f"{room:<15} {stats['total']:<12,} {stats['unique']:<12,} {stats['unique_pct']:<12.2f}")
        print("-" * 80)
    print("=" * 80)
    
    # Print time range report if available
    if time_range_report and any(time_range_report.values()):
        print("\n⏰ Time Range Report (Top 3 values per room/column):")
        print("=" * 100)
        for column_name, room_data in time_range_report.items():
            if room_data:
                print(f"\n📊 {column_name}")
                print("-" * 100)
                for room, value_time_ranges in room_data.items():
                    print(f"\n  {room.upper()}:")
                    
                    # Sort by count and show top 3
                    sorted_values = sorted(value_time_ranges.items(), 
                                         key=lambda x: x[1]['count'], 
                                         reverse=True)[:3]
                    
                    for value, time_info in sorted_values:
                        first_str = time_info['first'].strftime('%Y-%m-%d %H:%M:%S')
                        last_str = time_info['last'].strftime('%Y-%m-%d %H:%M:%S')
                        duration = time_info['last'] - time_info['first']
                        print(f"    Value {value:.2f}: {time_info['count']:,} occurrences")
                        print(f"      ⏰ {first_str} ~ {last_str}")
                        print(f"      ⏱️  Duration: {duration}")
                print("-" * 100)
        print("=" * 100)
    
    return fig

# Usage example
if __name__ == "__main__":
    # Set folder path
    # folder_path = "../../../data/kweather/version1_250804-250812"  # Change to your actual path
    folder_path = "../../../data/kweather/version2_250813"  # Change to your actual path
    
    # Optional: Specify time column name (will auto-detect if None)
    time_col = None  # e.g., 'timestamp', 'datetime', '시간' or None for auto-detect
    
    # 1. Separate plots for each room (3x2 grid)
    print("=" * 60)
    print("Separate Room Analysis (3x2 Grid)")
    print("=" * 60)
    fig1 = plot_column_distributions_separate(folder_path, "환기유효전력(W)", time_column=time_col)
    fig2 = plot_column_distributions_separate(folder_path, "냉난방유효전력(W)", time_column=time_col)
    
    # # 2. Side-by-side comparison (1x3 grid)
    # print("\n" + "=" * 60)
    # print("Side-by-side Room Comparison")
    # print("=" * 60)
    # fig3 = plot_column_comparison(folder_path, "환기유효전력(W)", time_column=time_col)
    
    # # 3. Multiple columns by room
    # print("\n" + "=" * 60)
    # print("Multiple Columns by Room")
    # print("=" * 60)
    # columns_to_analyze = ['환기유효전력(W)', '냉난방유효전력(W)']
    # fig4 = plot_multiple_columns_by_room(folder_path, columns_to_analyze, time_column=time_col)